In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# 固定随机种子，保证结果可复现
SEED = 77
rng = np.random.default_rng(SEED)

# 实验规模
N_PER_GROUP = 13_000
N = N_PER_GROUP * 2

# 实验参数
BASELINE_CONVERSION = 0.08
TREATMENT_CONVERSION = 0.09

# 数据保存目录
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"总用户数：{N:,}")
print(f"每组用户数：{N_PER_GROUP:,}")

总用户数：26,000
每组用户数：13,000


第 2 步：生成用户表 users.csv

In [2]:
# 随机分配实验组
group = np.array(
    ["A"] * N_PER_GROUP + ["B"] * N_PER_GROUP
)
rng.shuffle(group)

# 生成注册时间
# 模拟实验观察截止时间
observation_end = pd.Timestamp("2026-09-30 23:59:59")

# 用户入组时间：9月1日至9月22日
start_time = pd.Timestamp("2026-09-01")
end_time = pd.Timestamp("2026-09-23 00:00:00")

register_time = start_time + pd.to_timedelta(
    rng.random(N) * (end_time - start_time).total_seconds(),
    unit="s"
)

users = pd.DataFrame({
    "user_id": [f"U{i:06d}" for i in range(1, N + 1)],
    "register_time": register_time,
    "group": group,
    "channel": rng.choice(
        ["organic", "paid_ads", "social", "referral"],
        size=N,
        p=[0.40, 0.30, 0.20, 0.10]
    ),
    "device": rng.choice(
        ["iOS", "Android"],
        size=N,
        p=[0.45, 0.55]
    )
})

users = users.sort_values("register_time").reset_index(drop=True)

users.head()

,user_id,register_time,group,channel,device
0,U015574,2026-09-01 00:00:42.473674304,A,paid_ads,Android
1,U013049,2026-09-01 00:01:42.403466561,A,organic,iOS
2,U001700,2026-09-01 00:04:08.920057141,A,organic,iOS
3,U010194,2026-09-01 00:09:26.178828394,A,organic,Android
4,U009967,2026-09-01 00:09:56.749667194,A,social,Android


第 3 步：模拟行为事件
我们按用户逐步模拟：首页曝光、商品点击、加购和支付。

In [3]:
events_list = []
orders_list = []
activity_list = []

# 每位用户的购买结果
purchase_users = {}

for row in users.itertuples(index=False):
    uid = row.user_id
    grp = row.group
    reg_time = row.register_time

    # 所有入组用户均记录首页曝光
    events_list.append({
        "user_id": uid,
        "event_type": "home_exposure",
        "event_time": reg_time
    })

    # 实验组额外记录推荐模块曝光
    if grp == "B":
        events_list.append({
            "user_id": uid,
            "event_type": "recommendation_exposure",
            "event_time": reg_time
        })

    # 1. 商品点击
    click_prob = 0.40 if grp == "A" else 0.44
    clicked = rng.random() < click_prob

    if clicked:
        click_time = reg_time + pd.Timedelta(
            minutes=int(rng.integers(1, 60))
        )
        events_list.append({
            "user_id": uid,
            "event_type": "product_click",
            "event_time": click_time
        })

        # 2. 加购
        added = rng.random() < 0.50

        if added:
            cart_time = click_time + pd.Timedelta(
                minutes=int(rng.integers(1, 120))
            )
            events_list.append({
                "user_id": uid,
                "event_type": "add_to_cart",
                "event_time": cart_time
            })

            # 3. 支付：使两组理论转化率约为 8% 和 9%
            pay_prob = (
                0.40 if grp == "A"
                else TREATMENT_CONVERSION / (0.44 * 0.50)
            )

            paid = rng.random() < pay_prob

            if paid:
                order_time = cart_time + pd.Timedelta(
                    minutes=int(rng.integers(1, 180))
                )

                order_id = f"O{len(orders_list) + 1:07d}"
                amount = round(
                    float(rng.lognormal(mean=4.0, sigma=0.5)), 2
                )
                cost = round(amount * 0.65, 2)

                # 第一版先设两组退款概率相同
                refunded = rng.random() < 0.05
                refund_amount = amount if refunded else 0.0

                contribution_profit = round(
                    amount - cost - refund_amount, 2
                )

                orders_list.append({
                    "order_id": order_id,
                    "user_id": uid,
                    "order_time": order_time,
                    "amount": amount,
                    "cost": cost,
                    "refunded": int(refunded),
                    "refund_amount": refund_amount,
                    "contribution_profit": contribution_profit
                })

                events_list.append({
                    "user_id": uid,
                    "event_type": "purchase",
                    "event_time": order_time
                })

                purchase_users[uid] = order_time

    # 4. 模拟注册后 1-7 日的活跃行为
    # 此处的概率为模拟假设，后续可调整
    daily_activity_probs = [
        0.25, 0.20, 0.18, 0.16, 0.15, 0.13, 0.12
    ]

    for day, prob in enumerate(daily_activity_probs, start=1):
        active = rng.random() < prob

        if active:
            activity_time = (
                reg_time.normalize()
                + pd.Timedelta(days=day)
                + pd.Timedelta(
                    minutes=int(rng.integers(0, 1440))
                )
            )

            activity_list.append({
                "user_id": uid,
                "activity_day": day,
                "activity_date": activity_time.normalize()
            })

# 转换为 DataFrame
events = pd.DataFrame(events_list)
orders = pd.DataFrame(orders_list)
activity = pd.DataFrame(activity_list)

print("行为事件数：", len(events))
print("订单数：", len(orders))
print("活跃记录数：", len(activity))

行为事件数： 57649
订单数： 2236
活跃记录数： 30763


第四步：保存四张用户表

In [4]:
users.to_csv(DATA_DIR / "users.csv", index=False)
events.to_csv(DATA_DIR / "events.csv", index=False)
orders.to_csv(DATA_DIR / "orders.csv", index=False)
activity.to_csv(DATA_DIR / "activity.csv", index=False)

print("四张模拟数据表已保存。")

四张模拟数据表已保存。


验证数据有没有违反实验设计

In [6]:
# 1. 用户数量与分组比例
print(users["group"].value_counts())

# 2. 用户 ID 是否唯一
print("用户 ID 唯一：", users["user_id"].is_unique)

# 3. 订单 ID 是否唯一
print("订单 ID 唯一：", orders["order_id"].is_unique)

# 4. 每个用户是否最多一笔订单
print(
    "每个用户最多一笔订单：",
    orders["user_id"].is_unique
)

# 5. 每组的实际 7 日首单转化率
converted = set(orders["user_id"])

conversion_summary = (
    users.assign(
        converted=users["user_id"].isin(converted).astype(int)
    )
    .groupby("group")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
)

conversion_summary

group
A    13000
B    13000
Name: count, dtype: int64
用户 ID 唯一： True
订单 ID 唯一： True
每个用户最多一笔订单： True


,users,conversions,conversion_rate
group,,,
A,13000,1089,0.083769
B,13000,1147,0.088231


In [7]:
# 检查所有用户的7日观察窗口是否已成熟
users["observation_end"] = (
    users["register_time"] + pd.Timedelta(days=7)
)

print("最早注册时间：", users["register_time"].min())
print("最晚注册时间：", users["register_time"].max())
print("最晚观察窗口结束时间：", users["observation_end"].max())
print("统一观察截止时间：", observation_end)

assert (users["observation_end"] <= observation_end).all()

print("检查通过：所有用户均已完成完整的7日观察窗口。")

最早注册时间： 2026-09-01 00:00:42.473674304
最晚注册时间： 2026-09-22 23:58:46.694387766
最晚观察窗口结束时间： 2026-09-29 23:58:46.694387766
统一观察截止时间： 2026-09-30 23:59:59
检查通过：所有用户均已完成完整的7日观察窗口。
